In [7]:
import json
import os

folder = "."   # since your files are in LJ-ADS

files = [f for f in os.listdir(folder) if f.endswith(".geojson")]

print("GeoJSON Files Found:\n", files)
print("="*70)

for file in files:
    print(f"\n📁 File: {file}")
    print("-"*60)

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        data = json.load(f)

    features = data.get("features", [])

    print("Total Features:", len(features))

    if features:
        sample = features[0]

        print("\nSample Properties:")
        print(sample.get("properties", {}))

        print("\nGeometry Type:", sample["geometry"]["type"])
        print("Coordinates:", sample["geometry"]["coordinates"])

    print("="*70)

GeoJSON Files Found:
 ['Library.geojson', 'Municipal_gyms.geojson', 'Swimming_pools.geojson']

📁 File: Library.geojson
------------------------------------------------------------
Total Features: 62

Sample Properties:
{'Name': 'Chandrakant Baxi Library', 'description': None}

Geometry Type: Point
Coordinates: [72.566632, 23.06356, 0.0]

📁 File: Municipal_gyms.geojson
------------------------------------------------------------
Total Features: 39

Sample Properties:
{'Name': 'Khokhra Vyayam shala', 'description': 'Nearer to Khokhra Swimming pool,57-Khokhra ward,Ahmedabad'}

Geometry Type: Point
Coordinates: [72.616238, 22.9987, 0.0]

📁 File: Swimming_pools.geojson
------------------------------------------------------------
Total Features: 14

Sample Properties:
{'Name': 'Vir Savarkar Muni.Swimmingpool', 'description': 'Vasna,Opp. Godawari Flats, B/h. Anjali Cinema, Vasna'}

Geometry Type: Point
Coordinates: [72.549721, 23.004688, 0.0]


In [9]:
import json
import os
import pandas as pd

folder = "."
rows = []

files = [f for f in os.listdir(folder) if f.endswith(".geojson")]

for file in files:
    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        data = json.load(f)

    for feat in data["features"]:
        geom = feat["geometry"]
        props = feat.get("properties", {})

        if geom["type"] == "Point":
            coords = geom["coordinates"]

            lon = coords[0]
            lat = coords[1]

            rows.append({
                "name": props.get("Name", props.get("name", "Unknown")),
                "category": file.replace(".geojson", ""),
                "lat": lat,
                "lon": lon
            })

df = pd.DataFrame(rows)

print("Total extracted points:", len(df))
print(df.head())

df.to_csv("ahmedabad_facilities.csv", index=False)

Total extracted points: 115
                                  name category        lat        lon
0             Chandrakant Baxi Library  Library  23.063560  72.566632
1  Kavi ramanlal vasantlal Desai,Nikol  Library  23.025530  72.643914
2            V.N.Shah Library,Rakhiyal  Library  23.021774  72.623712
3                   B.N.Library,Raipur  Library  23.021184  72.594867
4           Dr.Shyama Prashad Mukharji  Library  23.011051  72.531455


In [4]:
import pandas as pd
import folium

df = pd.read_csv("ahmedabad_facilities.csv")

m = folium.Map(location=[23.0225, 72.5714], zoom_start=12)

icon_map = {
    "Library": ("blue", "book"),
    "Municipal_gyms": ("green", "glyphicon glyphicon-heart"),
    "Swimming_pools": ("purple", "tint")
}

for _, row in df.iterrows():
    cat = row["category"]
    
    color, icon = icon_map.get(cat, ("red", "info-sign"))

    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=f"{row['name']} ({cat})",
        icon=folium.Icon(color=color, icon=icon)
    ).add_to(m)

m